In [2]:
import pandas as  pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
import keras_tuner as kt


In [3]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline

In [5]:
data = pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\churn_prediction\Data\Churn_Modelling.xls")

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [6]:
data.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
x = data.drop(columns=['RowNumber','CustomerId', 'Surname', 'Exited'])
y = data['Exited']

In [8]:
x = pd.get_dummies(x ,columns=['Geography', 'Gender'], drop_first= True, dtype= int)

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state= 42, test_size= 0.2)


In [10]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state= 42)
x_train_tra, y_train = smote.fit_resample(x_train, y_train)

In [154]:
from sklearn.utils import shuffle

x_train_tra, y_train = shuffle(x_train_tra, y_train, random_state=42)


In [16]:
scaler = StandardScaler()
x_train_tras = scaler.fit_transform(x_train_tra)
x_test_tras = scaler.transform(x_test)

In [172]:
x_test_tras.shape

(2000, 11)

## Multimodel apply and combine performance check

In [55]:
# XGBoost model
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    tree_method='hist',
    scale_pos_weight=4,
    eval_metric='auc'
)

# Logistic Regression model
lr_model = LogisticRegression(max_iter=500, class_weight='balanced')

# SVM model
svc = SVC(class_weight='balanced')

# Random Forest Classifier model
rfc = RandomForestClassifier(class_weight='balanced')
 
# ANN model
ann_model = Sequential()
ann_model.add(Dense(64, activation='relu', input_shape=(x_train_tras.shape[1],)))

ann_model.add(Dense(32, activation='relu'))

ann_model.add(Dense(5, activation='relu'))

ann_model.add(Dense(1, activation='sigmoid'))
ann_model.compile(optimizer=tensorflow.keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9), loss='binary_crossentropy', metrics=['accuracy'])


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
# Train XGBoost
xgb_model.fit(x_train_tras, y_train)

# Train Logistic Regression
lr_model.fit(x_train_tras, y_train)

# Train SVM 
svm_model = svc.fit(x_train_tras, y_train)

# Train Random Forest Classifier 
rfc_model = rfc.fit(x_train_tras, y_train)
# Train ANN
ann_model.fit(x_train_tras, y_train, epochs=20, batch_size=32, verbose=0)


In [57]:
xgb_pred = xgb_model.predict(x_test_tras)
lr_pred = lr_model.predict(x_test_tras)
svm_pred  = svm_model.predict(x_test_tras)
rfc_pred  = rfc_model.predict(x_test_tras)
ann_pred = (ann_model.predict(x_test_tras) > 0.3).astype(int).flatten()

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [58]:
preds = np.vstack([xgb_pred, lr_pred, ann_pred,svm_pred, rfc_pred ]).T

final_pred = []
for row in preds:
    counts = np.bincount(row)
    final_pred.append(np.argmax(counts))

final_pred = np.array(final_pred)


In [59]:
from sklearn.metrics import classification_report, confusion_matrix
print("accuracy_score :- ", accuracy_score(y_test, final_pred))
print("Final Ensemble Report:\n", classification_report(y_test, final_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, final_pred))


accuracy_score :-  1.0
Final Ensemble Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1607
           1       1.00      1.00      1.00       393

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Confusion Matrix:
 [[1607    0]
 [   0  393]]


### Only XGBOOST model apply with hyperameter to check the model performance


In [11]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    tree_method='hist',
    scale_pos_weight=4,
    eval_metric='auc'
)

In [13]:
xgb_model.fit(x_train_tra, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [15]:
ypred = xgb_model.predict(x_test)

print("accuracy_score :- ", accuracy_score(y_test, ypred))
print("accuracy_score :- ", confusion_matrix(y_test, ypred))

accuracy_score :-  0.767
accuracy_score :-  [[1247  360]
 [ 106  287]]


### ANN best Model apply using keras tuner 

In [17]:
def model_builder(hp):
  model = keras.Sequential()
  count = 0

  for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):
    if count == 0:
      unite = hp.Int('num_node' + str(i), min_value= 8, max_value = 128, step = 8)
      activation_fun = hp.Choice('activation_' + str(i), values = ['relu', 'tanh', 'selu', 'elu'])
      # dropout = hp.Float('dropout_' + str(i), min_value = 0.1, max_value = 0.9, step = 0.1)

      model.add(
        Dense(units=unite,
              activation= activation_fun, 
              input_dim = 11
            )
        )
      # model.add(Dropout(rate=dropout))

    else:
        unite = hp.Int('num_node' + str(i), min_value= 8, max_value = 128, step = 8)
        activation_fun = hp.Choice('activation_' + str(i), values = ['relu', 'tanh', 'selu', 'elu'])
        model.add(
            Dense(units=unite,
                activation= activation_fun
                )
              )
        # model.add(Dropout(rate=dropout))

    count+= 1
      

  model.add(Dense(1, activation= 'sigmoid'))
  model.compile(optimizer= hp.Choice('optimizer', values = ['adam', 'rmsprop', 'sgd', 'nadam', 'adadelta']),
                loss= 'binary_crossentropy',
                metrics=['accuracy'])

  return model

In [18]:
tuner = kt.RandomSearch(model_builder,
                     objective='val_accuracy',
                     max_trials=5,
                   
                     directory='hello -6',
                     project_name='my -6')


c:\Users\Admin\OneDrive\Desktop\churn_prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
tuner.search(x_train_tras, y_train, epochs=5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 10s]
val_accuracy: 0.7055000066757202

Best val_accuracy So Far: 0.7055000066757202
Total elapsed time: 00h 00m 49s


In [176]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 9,
 'num_node0': 128,
 'activation_0': 'selu',
 'optimizer': 'rmsprop',
 'num_node1': 8,
 'activation_1': 'relu',
 'num_node2': 8,
 'activation_2': 'relu',
 'num_node3': 8,
 'activation_3': 'relu',
 'num_node4': 8,
 'activation_4': 'relu',
 'num_node5': 8,
 'activation_5': 'relu',
 'num_node6': 8,
 'activation_6': 'relu',
 'num_node7': 8,
 'activation_7': 'relu',
 'num_node8': 8,
 'activation_8': 'relu'}

In [20]:
model =tuner.get_best_models(num_models=1)[0]
stop_early = tensorflow.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7)
model.fit(x_train_tras, y_train, epochs = 200, initial_epoch = 6, validation_data = (x_test_tras, y_test))

c:\Users\Admin\OneDrive\Desktop\churn_prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 7/200


c:\Users\Admin\OneDrive\Desktop\churn_prediction\venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adadelta', because it has 2 variables whereas the saved optimizer has 30 variables. 
  saveable.load_own_variables(store)


398/398 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7422 - loss: 0.6348 - val_accuracy: 0.6840 - val_loss: 0.6519
Epoch 8/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7516 - loss: 0.6258 - val_accuracy: 0.6990 - val_loss: 0.6429
Epoch 9/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7588 - loss: 0.6166 - val_accuracy: 0.7070 - val_loss: 0.6341
Epoch 10/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7648 - loss: 0.6073 - val_accuracy: 0.7190 - val_loss: 0.6256
Epoch 11/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7688 - loss: 0.5980 - val_accuracy: 0.7240 - val_loss: 0.6173
Epoch 12/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7707 - loss: 0.5890 - val_accuracy: 0.7295 - val_loss: 0.6094
Epoch 13/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7738 - loss: 0.5802 - val_accuracy: 0.7330 - val_loss: 0.6017
Epoch 14/200
398/398 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7751 - loss: 0.5716 - val_accuracy: 0

In [21]:
yprob = model.predict(x_test_tras)
ypred = np.where(yprob>0.5, 1,0)
print("accuracy_score :- ", accuracy_score(y_test, ypred))
print("accuracy_score :- ", confusion_matrix(y_test, ypred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
accuracy_score :-  0.7925
accuracy_score :-  [[1340  267]
 [ 148  245]]
